# COMP813 Project — Underwater Plastic Bags vs Medusozoa (local Jupyter version)
Runs the project scripts on your own laptop and GPU (no Colab, no Google Drive, no zip files).

**Expected folder layout** (upper/lower case does not matter):
```
Desktop/AUT/AI/COMP813/
    project_code/     <- run_all.py, prepare_data.py, requirements.txt, ...
    outputs/          <- results are written here
    Task/
        archive_jelly/
        archive_trashcan/
        archive_Plastic/
```
**Just run the cells from top to bottom.** Finished experiments are skipped, so after a crash, a sleep or a closed laptop you can simply run the cell again (run sections 0 and 0b first).

Tip: run the notebook with Jupyter started from your normal Python/conda environment, and keep the laptop plugged in and set to *never sleep* for the long runs.

> **Note.** Cell outputs are cleared in this public version (they contained local paths and dataset images). The result files of the full runs (local RTX 4050 GPU, seeds 813–815) are in `results_snapshot/`.

## 0. Find the folders and set up helpers

In [ ]:
import os, sys, json, glob, subprocess, codecs, shutil, time, re
from pathlib import Path
from IPython.display import Image, Markdown, display

# ---- Change this ONLY if your main folder is somewhere else ---------------------------
BASE_DIR = None          # e.g. BASE_DIR = r"D:\Uni\COMP813"   (None = look on the Desktop)
# ----------------------------------------------------------------------------------------

def find_base():
    if BASE_DIR:
        return Path(BASE_DIR).expanduser()
    home = Path.home()
    desktops = [home / "Desktop", home / "OneDrive" / "Desktop", home / "OneDrive - AUT University" / "Desktop"]
    desktops += list(home.glob("OneDrive*/Desktop"))
    for d in desktops:
        p = d / "AUT" / "AI" / "COMP813"
        if p.exists():
            return p
    raise FileNotFoundError("Could not find Desktop/AUT/AI/COMP813. Set BASE_DIR in this cell.")

def child(parent, name):
    """Find a sub-folder ignoring upper/lower case."""
    for p in parent.iterdir():
        if p.is_dir() and p.name.lower() == name.lower():
            return p
    return None

BASE = find_base()

# code folder = the sub-folder that contains run_all.py (works for project_code / COMP813_Project_Code ...)
CODE_DIR = next((p for p in sorted(BASE.iterdir()) if p.is_dir() and (p / "run_all.py").exists()), None)
if CODE_DIR is None:                                   # maybe one level deeper (unzipped twice)
    CODE_DIR = next((r.parent for r in BASE.glob("*/*/run_all.py")), None)
if CODE_DIR is None:
    raise FileNotFoundError(f"run_all.py not found inside {BASE}. Folders there: {[p.name for p in BASE.iterdir()]}")

DATA_ROOT = child(BASE, "Task") or child(BASE, "Tasks")
if DATA_ROOT is None:
    raise FileNotFoundError(f"No 'Task' folder inside {BASE}")
OUT_DIR = child(BASE, "outputs") or (BASE / "outputs")
OUT_DIR.mkdir(exist_ok=True)

# The project scripts read these two environment variables
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["OUT_DIR"] = str(OUT_DIR)

ENV = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8", PYTHONUTF8="1",
           KMP_DUPLICATE_LIB_OK="TRUE",   # avoids the Windows "two OpenMP libraries" crash (common with Anaconda)
           PYTHONFAULTHANDLER="1")        # prints where a native crash happened

def run(*args):
    """Run a project script with THIS notebook's Python, inside the code folder, and stream its output."""
    cmd = [sys.executable, "-u", *map(str, args)]
    print("$", " ".join(cmd), f"   (in {CODE_DIR.name})\n")
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(CODE_DIR), env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    tail = ""
    try:
        while True:
            chunk = proc.stdout.read1(4096)
            if not chunk:
                break
            txt = dec.decode(chunk); tail = (tail + txt)[-2500:]
            print(txt, end="", flush=True)
    except KeyboardInterrupt:                         # the Stop button in Jupyter
        proc.terminate(); proc.wait()
        print("\n[stopped - run the cell again to continue; finished runs are skipped]")
        raise
    rc = proc.wait()
    if rc != 0:
        hint = ""
        if "3221226505" in tail or "3221225477" in tail:
            hint = ("\n\nHINT: this is a silent Windows crash (no Python error). Run the same cell again - finished runs are skipped. "
                    "If it crashes at the same place, run the diagnostic cell in section 3.\n")
        raise RuntimeError(f"Command failed with exit code {rc}.{hint} LAST OUTPUT OF THE SCRIPT:\n\n{tail}")
    m = (time.time() - t0) / 60
    print(f"\n[finished in {m:.1f} min = {m/60:.2f} h]")

print("Main folder :", BASE)
print("Code folder :", CODE_DIR)
print("Data (Task) :", DATA_ROOT)
print("Outputs     :", OUT_DIR)
print("Python      :", sys.executable)

In [ ]:
run("run_all.py", "--stages", "main", "underwater", "--seeds", "814", "815")

### 0b. Check that the three datasets are where the code expects them

In [ ]:
def check(label, ok, where=""):
    print(("OK      " if ok else "MISSING ") + label, where)
    return ok

jelly = child(DATA_ROOT, "archive_jelly")
tcan  = child(DATA_ROOT, "archive_trashcan")
plast = child(DATA_ROOT, "archive_Plastic")

results = [
    check("archive_jelly",     jelly is not None),
    check("  Train_Test_Valid", jelly is not None and any(p.name.lower() == "train_test_valid" for p in jelly.iterdir())),
    check("archive_trashcan",  tcan is not None),
    check("  instances_train_trashcan.json", tcan is not None and any(tcan.rglob("instances_train_trashcan.json"))),
    check("  instances_val_trashcan.json",   tcan is not None and any(tcan.rglob("instances_val_trashcan.json"))),
    check("archive_Plastic",   plast is not None),
    check("  Plastic Bag Images", plast is not None and any(plast.rglob("Plastic Bag Images"))),
    check("  Garbage Bag Images", plast is not None and any(plast.rglob("Garbage Bag Images"))),
]
print()
print("Everything found - continue." if all(results) else
      "Fix the MISSING items above (folder names must be exactly as in PROCESS.md, Step 1) before continuing.")

## 1. Install the libraries and check the GPU
* Installs `requirements.txt` into the Python that runs this notebook.
* On Windows, `pip` often installs the **CPU-only** PyTorch. If that happens and an NVIDIA GPU is found, the cell installs the CUDA build automatically.
* Change `CUDA_TAG` if needed (`cu118`, `cu121`, `cu124`, `cu126`). Pick one that is not newer than the CUDA version shown by `nvidia-smi`; any recent NVIDIA driver works with `cu124`.

In [ ]:
CUDA_TAG = "cu124"

def cuda_ok():
    r = subprocess.run([sys.executable, "-c", "import torch;print(torch.cuda.is_available())"],
                       capture_output=True, text=True, env=ENV)
    return r.stdout.strip() == "True"

def pip(*a):
    run("-m", "pip", "install", *a)

pip("-q", "-r", "requirements.txt")

if not cuda_ok():
    if shutil.which("nvidia-smi"):
        print("\nPyTorch cannot see the GPU -> installing the CUDA build of PyTorch (large download)...\n")
        pip("--force-reinstall", "torch", "torchvision", "--index-url", f"https://download.pytorch.org/whl/{CUDA_TAG}")
    else:
        print("\nnvidia-smi not found: no NVIDIA driver/GPU detected. Install the driver, then re-run this cell.")

r = subprocess.run([sys.executable, "-c",
    "import torch;print('torch', torch.__version__);ok=torch.cuda.is_available();print('GPU available:', ok);"
    "print(torch.cuda.get_device_name(0) if ok else 'NO GPU - training will be extremely slow')"],
    capture_output=True, text=True, env=ENV)
print(r.stdout or r.stderr)

# ---- CLIP fix: transformers >= 4.52 refuses to load CLIP's .bin weights with torch < 2.6 ----
def pkg_version(pkg):
    r = subprocess.run([sys.executable, "-c", f"import importlib.metadata as m;print(m.version('{pkg}'))"],
                       capture_output=True, text=True, env=ENV)
    return tuple(int(x) for x in re.findall(r"\d+", r.stdout.split("+")[0])[:2]) or (0,)

torch_v, tf_v = pkg_version("torch"), pkg_version("transformers")
print("torch", torch_v, "| transformers", tf_v)
if torch_v < (2, 6) and tf_v >= (4, 52):
    print("\ntorch < 2.6 with a new transformers: CLIP cannot be loaded -> switching transformers to 4.51.3 (small download)\n")
    pip("-q", "transformers==4.51.3")
    print("transformers now:", pkg_version("transformers"))
else:
    print("CLIP loading check: OK")

### 1b. Make sure the GPU is really used
This cell (a) runs a real calculation on the GPU, (b) shows which lines of your scripts choose the GPU, and (c) **stops with an error if PyTorch cannot use the GPU**, so you never start a multi-hour run on the CPU by accident.

While an experiment is running you can watch the GPU: open a terminal and type `nvidia-smi -l 2`, or on Windows open Task Manager -> Performance -> your NVIDIA GPU and switch one of the graphs to **Cuda**. During training, GPU utilisation should be well above 0 % and memory used should go up.

In [ ]:
test = ("import torch, time;"
        "assert torch.cuda.is_available(), 'CUDA NOT AVAILABLE';"
        "d=torch.device('cuda');x=torch.randn(4000,4000,device=d);torch.cuda.synchronize();t=time.time();"
        "[x@x for _ in range(20)];torch.cuda.synchronize();"
        "print('GPU:',torch.cuda.get_device_name(0));"
        "print('GPU memory: %.1f GB'%(torch.cuda.get_device_properties(0).total_memory/1e9));"
        "print('20 matrix multiplications took %.2f s on the GPU'%(time.time()-t))")
r = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True, env=ENV)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-1500:])
    raise RuntimeError("PyTorch cannot use the GPU. Re-run section 1 (CUDA build of PyTorch) and update your NVIDIA driver.")

print("Lines in your scripts that select the device:")
hits = 0
for f in sorted(CODE_DIR.glob("*.py")):
    for n, line in enumerate(f.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
        if "cuda" in line.lower() and hits < 15:
            print(f"  {f.name}:{n}: {line.strip()[:110]}"); hits += 1
if hits == 0:
    print("  (none found - the scripts may not use the GPU! Send me train.py and I will check.)")
print()
subprocess.run(["nvidia-smi"])

## 2. Prepare the data
Cleans the data, cuts the bags out of the TrashCan video frames, splits 60/20/20 without leakage, builds the extra test sets and checks the class balance. Only needed once (about 1 minute).

In [ ]:
run("prepare_data.py")

s = json.load(open(OUT_DIR / "data_summary.json", encoding="utf-8"))
print(json.dumps({k: s[k] for k in ["jellyfish_duplicates", "trashcan_official_split", "trashcan_bag_cleaning"]}, indent=1))
for task in ["binary", "fine"]:
    for split in ["train", "train_unique_photos_only", "test"]:
        b = s["class_balance"][task][split]
        print(task, split, b["counts"], "imbalance ratio %.2f" % b["imbalance_ratio"])

In [ ]:
for f in ["data_class_balance", "data_samples", "data_trashcan_cleaning", "data_underwater_severity", "data_background_swap_examples"]:
    display(Image(filename=str(OUT_DIR / "figures" / f"{f}.png")))

## 3. Quick smoke test (few images, 1 epoch)
The numbers mean nothing; it only checks that every script runs on your machine (roughly 10-15 minutes on a laptop GPU). Good for the video demo too.

In [ ]:
run("run_all.py", "--quick")

### 3b. Only if the smoke test crashed with exit code 3221226505 (silent Windows crash)
First simply **run the cell above again** - it resumes where it stopped. If it dies again at the same script, run this cell: it runs the crashing step on its own and prints the place where the crash happens. Send me that output.

In [ ]:
run("-X", "faulthandler", "robustness.py", "--runs", "QUICK_binary_resnet50_pt_ft_aug-standard_s813", "--quick")
print(); subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,temperature.gpu", "--format=csv"])


## 4. Full experiments (all seeds, one after another)
Run the cells in order. Each one can be stopped (Stop button) and re-run later - finished runs are skipped. To run **everything** in one go, use the last cell of this section instead.

**Seeds:** you are working alone, so the default is **one seed (813)**, about a third of the time. The results then have no real "mean +- std". Later you can set `SEEDS = ["813", "814", "815"]` and re-run the cells; the finished seed-813 runs are skipped and only the new seeds are computed.
Each cell prints how long it took, so after 4b you know how long the rest will need.

In [ ]:
SEEDS = ["813"]          # add "814", "815" later if you have time

In [ ]:
# 4a: classic baselines, CLIP and the learning-rate search
run("run_all.py", "--stages", "classic", "clip", "lrsweep")

In [ ]:
# 4b: main models + underwater-domain augmentation (uses SEEDS)
run("run_all.py", "--stages", "main", "underwater", "--seeds", *SEEDS)


In [ ]:
# 4c: class-imbalance methods (uses SEEDS)
run("run_all.py", "--stages", "imbalance", "--seeds", *SEEDS)

In [ ]:
# 4d: cross-dataset study (train on synthetic / both bag sources)
run("run_all.py", "--stages", "crossdata", "--seeds", *SEEDS)

In [ ]:
# 4e: ablations and data-size study
run("run_all.py", "--stages", "ablation", "datasize", "--seeds", *SEEDS)

In [ ]:
# OPTIONAL - everything from 4a to 4e in a single command (skip this if you ran the cells above)
# run("run_all.py", "--seeds", *SEEDS)

## 5. Grad-CAM and summary (after all results are in `outputs/`)

### 5A 
GPU allocation

In [ ]:
from pathlib import Path
import shutil

f = CODE_DIR / "gradcam.py"          # CODE_DIR is defined in section 0 of the notebook
src = f.read_text(encoding="utf-8")
if "GRADCAM_MEMORY_PATCH" in src:
    print("gradcam.py is already patched.")
else:
    shutil.copy(f, f.with_suffix(".py.bak"))
    edits = [
        ("import argparse\n",
         "import argparse\nimport gc  # GRADCAM_MEMORY_PATCH\n"),
        ("        layer.register_forward_hook(self._save)",
         "        self.handle = layer.register_forward_hook(self._save)"),
        ("        return cam.detach().cpu().numpy(), F.softmax(logits.detach().float(), 1).cpu().numpy()",
         "        out = (cam.detach().cpu().numpy(), F.softmax(logits.detach().float(), 1).cpu().numpy())\n"
         "        self.acts = self.grads = None   # free the GPU memory held by the last forward pass\n"
         "        return out"),
        ("def analyse_run(ckpt_path, df, n_fig=4, batch=16):",
         "def analyse_run(ckpt_path, df, n_fig=4, batch=4):"),
        ('    model, ck = load_checkpoint(ckpt_path)\n    names = C.class_names(ck["task"])\n    cammer',
         '    model, ck = load_checkpoint(ckpt_path)\n    for p_ in model.parameters():\n        p_.requires_grad_(False)   # only the input needs gradients -> much less memory\n'
         '    names = C.class_names(ck["task"])\n    cammer'),
        ('    fig.tight_layout(); fig.savefig(out_fig / f"{run}_test_examples.png", dpi=130); plt.close(fig)\n    return res',
         '    fig.tight_layout(); fig.savefig(out_fig / f"{run}_test_examples.png", dpi=130); plt.close(fig)\n    cammer.handle.remove()\n    return res'),
        ("        all_res.append(analyse_run(ck, df))",
         "        all_res.append(analyse_run(ck, df))\n        gc.collect(); torch.cuda.empty_cache()   # release this model before loading the next one"),
    ]
    for old, new in edits:
        assert src.count(old) == 1, f"Could not find this text in gradcam.py:\n{old}"
        src = src.replace(old, new)
    f.write_text(src, encoding="utf-8")
    print("gradcam.py patched (backup: gradcam.py.bak).")


### 5B Run GRAD CAM and summary

In [ ]:
run("run_all.py", "--stages", "gradcam", "summary")

In [ ]:
display(Markdown((OUT_DIR / "summary" / "key_numbers.md").read_text(encoding="utf-8")))

In [ ]:
files = sorted(glob.glob(str(OUT_DIR / "summary" / "*.png"))) + \
        sorted(glob.glob(str(OUT_DIR / "figures" / "gradcam" / "*s813_background_swap.png")))[:3]
for f in files:
    print(f)
    display(Image(filename=f))